Note: Write your code in the code cells, and your responses in markdown. 
Run the entire script and display the outputs of your code. 

Due: **11:59PM Central Time on Monday, 11/03**. Upload both your code (.ipynb) and responses (html or pdf) to Canvas by then. 

In [ ]:
# --- PS5 setup & data load (robust paths, env check) ---
import sys, os
from pathlib import Path
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf  # <- formulas

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

print("Python:", sys.version.split()[0])
print("Kernel executable:", sys.executable)
print("CWD:", Path.cwd())

# Resolve project root (works in VS Code without manual `cd`)
nb_dir = Path.cwd()
project_root = nb_dir

# Candidate locations for the CSV
candidates = [
    project_root / "twins.csv",                       # same folder as notebook
    project_root / "Input" / "twins.csv",             # PS5/Input next to notebook
    project_root.parent / "Input" / "twins.csv",      # notebook in PS5/, data in PS5/Input/
    project_root / ".." / "Input" / "twins.csv",      # safety for odd launches
]
candidates = [p.resolve() for p in candidates]

csv_path = next((p for p in candidates if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError(
        "Could not find twins.csv.\nTried:\n  - " + "\n  - ".join(map(str, candidates))
    )

print("Using data file:", csv_path)

twins = pd.read_csv(csv_path)

print("Rows, Cols:", twins.shape)
display(twins.head())

# NOTE from 10/29
- i = family
- t = twin

# Princeton Twins Data
In this problem set we will fit a few models to the Princeton Twins Survey data. The data set is called twins.csv. The variables are:
- famid = family id variable
- t=1,2 for twin #1 or twin #2
- age = age (some observations have coded in part year values)
- educ = education
- oeduc = education of twin
- lw = log wage
- married = dummy 1 if married 0 if not
- omarried = dummy if twin is married
- female = 1 if female
- ofemale = 1 if twin is female.
- exp = “labor market experience” = age - educ - 6
- oexp = experience of twin

NOTE: all twins are one of two identical twins in the data set. So female=ofemale in all cases.

In [ ]:
# Load dataset (using pandas "pd")
twins = pd.read_csv("twins.csv") # add your own directory if necessary
print(twins.head(6))

# verify the data is unique at (famid, t) level, where t = 1,2 for twin #1 or #2
assert 0==twins[['famid','t']].duplicated().sum()

# verify twins have the same sex: 
assert (twins['female']==twins['ofemale']).all()

Before that going into Q1, we want to just do a few data checks detailed in the following cell.

In [ ]:
# --- Features & quick data checks ---
twins = twins.copy()

# experience^2
twins["exp2"] = twins["exp"]**2

# quick sanity checks used later
assert twins[["famid","t"]].duplicated().sum()==0, "famid,t should be unique (one row per twin)."
assert (twins["female"]==twins["ofemale"]).all(), "All pairs are same-sex by design."

twins[["lw","educ","exp","exp2","married","female"]].describe().T

## 1. OLS
Estimate a simple model relating log wages to: education, experience, experience-squared, married status, and female. 

In [ ]:
# --- OLS with HC1 robust SEs ---
import statsmodels.formula.api as smf
import pandas as pd

formula_q1 = "lw ~ educ + exp + exp2 + married + female"
ols_q1 = smf.ols(formula_q1, data=twins).fit(cov_type="HC1")

# Tidy table (coef, robust se, t, p, N, R2)
tbl_q1 = (
    pd.DataFrame({
        "coef": ols_q1.params,
        "se(HC1)": ols_q1.bse,
        "t": ols_q1.tvalues,
        "p>|t|": ols_q1.pvalues
    })
    .round(4)
)
display(tbl_q1)
print(f"N = {int(ols_q1.nobs)}, R2 = {ols_q1.rsquared:.4f}, R2_adj = {ols_q1.rsquared_adj:.4f}")

### Q1. Baseline OLS Regression: Log Wages and Individual Characteristics

We estimate the model  


$\text{lw}_i = \beta_0 + \beta_1 \text{educ}_i + \beta_2 \text{exp}_i + \beta_3 \text{exp}_i^2 + \beta_4 \text{married}_i + \beta_5 \text{female}_i + u_i$

using heteroskedasticity-robust (HC1) standard errors.

| Variable | Coefficient | Robust SE | t | p |  
|-----------|--------------|------------|------|------|  
| Intercept | 0.3455 | 0.1386 | 2.49 | 0.013 |  
| Education | 0.1238 | 0.0086 | 14.36 | 0.000 |  
| Experience | 0.0523 | 0.0048 | 10.88 | 0.000 |  
| Experience² | −0.0009 | 0.0001 | −8.28 | 0.000 |  
| Married | 0.0693 | 0.0321 | 2.16 | 0.031 |  
| Female | −0.3341 | 0.0309 | −10.83 | 0.000 |  
| N = 1,074   R² = 0.3146   R²_adj = 0.3113 |

**Interpretation.**  
Education has a strong, positive, and statistically significant effect on log wages: each additional year of schooling raises expected log wages by about $0.124$, or roughly $12.4 \%$.  
Experience also increases wages but at a diminishing rate: the positive coefficient on $exp$ ($0.052$) and negative coefficient on $exp^2$ ($−0.0009$) imply a concave experience–wage profile. The implied turning point is approximately:


$\text{exp}^* = \frac{0.0523}{2(0.0009)} \approx 29 \text{ years}$

beyond which additional experience yields little wage growth.

Being married is associated with about a $6.9 \%$ higher wage, statistically significant at $5 \% $. In contrast, the coefficient on female $(−0.334)$ indicates that, holding education and experience constant, women earn approximately $33 \% $ lower wages than men on average.

The model explains about $31 \% $ of the variation in log wages ($R^2 = 0.3146$), consistent with a well-fitting cross-sectional wage regression.

## 2. Separate models for men and women
Fit the same model separately for men and women. Does the model look different?
- Note: drop "female" from the regression because we are estimating the regression separately by gender. 

In [ ]:
# --- Q2: Separate OLS regressions for men and women (HC1 robust) ---

formula_q2 = "lw ~ educ + exp + exp2 + married"

# Separate by gender
men = twins.loc[twins["female"] == 0]
women = twins.loc[twins["female"] == 1]

# Fit OLS models with robust SEs
ols_men = smf.ols(formula_q2, data=men).fit(cov_type="HC1")
ols_women = smf.ols(formula_q2, data=women).fit(cov_type="HC1")

# Tidy summary table function
def tidy_ols(result, label):
    import pandas as pd
    tbl = (
        pd.DataFrame({
            "coef": result.params,
            "se(HC1)": result.bse,
            "t": result.tvalues,
            "p>|t|": result.pvalues
        })
        .round(4)
    )
    print(f"{label} results — N={int(result.nobs)}, R²={result.rsquared:.4f}, R²_adj={result.rsquared_adj:.4f}")
    display(tbl)
    return tbl

# Display results
tbl_m = tidy_ols(ols_men, "Men")
tbl_f = tidy_ols(ols_women, "Women")

We also calculate and display the turning points for each gender group as follows:

In [ ]:
turn_men = ols_men.params["exp"] / (-2 * ols_men.params["exp2"])
turn_women = ols_women.params["exp"] / (-2 * ols_women.params["exp2"])
print(f"Turning point (men): {turn_men:.2f} years")
print(f"Turning point (women): {turn_women:.2f} years")

### Q2. Separate Models for Men and Women

We now estimate the following specification separately by gender:

$\text{lw}_i = \beta_0 + \beta_1 \text{educ}_i + \beta_2 \text{exp}_i + \beta_3 \text{exp}_i^2 + \beta_4 \text{married}_i + u_i$

using heteroskedasticity-robust (HC1) standard errors.

| Variable | Coefficient (Men) | Coefficient (Women) |  
|-----------|------------------:|------------------:|  
| Intercept | 0.5197 | −0.0854 |  
| Education ($\beta_1$) | 0.1027 | 0.1369 |  
| Experience ($\beta_2$) | 0.0588 | 0.0529 |  
| Experience² ($\beta_3$) | −0.0010 | −0.0010 |  
| Married ($\beta_4$) | 0.1993 | −0.0399 |  
| N | 500 | 574 |  
| R² | 0.2553 | 0.3033 |  
| R²_adj | 0.2493 | 0.2984 |

**Interpretation.**  
The education coefficient remains positive and statistically significant for both genders, indicating that additional schooling increases log wages. The estimated effect is somewhat larger for women ($\beta_1 = 0.137$) than for men ($\beta_1 = 0.103$), suggesting that women experience slightly higher percentage returns to education conditional on being in the labor market.

Experience and experience-squared retain their expected signs: wages rise with experience at a diminishing rate. The concave wage–experience profiles imply turning points at approximately:

$\text{exp}^*_{men} = 29.8 \text{ years}, \quad \text{exp}^*_{women} = 26.9 \text{ years}$

That is, wage growth tends to level off around 30 years of experience for men and slightly earlier for women.

The marriage coefficient differs sharply by gender. For men, being married is associated with approximately a $19.9\%$ higher wage, statistically significant at the $1\%$ level, consistent with the male marriage premium.

For women, however, the coefficient on $married$ is small, negative ($−0.040$), and statistically insignificant, implying no wage premium (and possibly a mild penalty) for married women.

Finally, the model fits slightly better for women ($R^2 = 0.303$) than for men ($R^2 = 0.255$), indicating that the included covariates explain a bit more of the variation in women’s wages. Overall, the gender-specific regressions reinforce patterns commonly observed in labor economics: higher returns to education for women, but a wage premium from marriage primarily accruing to men.

## 3. Add mean family marriage rate
Construct the mean family marriage rate for each person in the data set (i.e., the fraction of the twins that is married, which can be 0, 1/2, or 1).

In [ ]:
# --- Q3(a): Construct mean family marriage rate ---
twins = twins.copy()

# Mean family marriage rate = average 'married' per famid
twins["mean_married"] = twins.groupby("famid")["married"].transform("mean")

# Check distinct possible values (should be 0, 0.5, 1)
print("Unique family marriage rates:", twins["mean_married"].unique())

# Quick sanity table: family id, t, married, mean_married
display(twins[["famid","t","married","mean_married"]].head(8))

### (a) 
Verify that when you regress marriage of a twin on the average fraction of the siblings who are married, you get a coefficient of 1.

In [ ]:
# --- Verify auxiliary regression: married_i ~ mean_married_fam ---
import statsmodels.formula.api as smf

aux_reg = smf.ols("married ~ mean_married", data=twins).fit(cov_type="HC1")

tbl_aux = (
    pd.DataFrame({
        "coef": aux_reg.params,
        "se(HC1)": aux_reg.bse,
        "t": aux_reg.tvalues,
        "p>|t|": aux_reg.pvalues
    })
    .round(4)
)

display(tbl_aux)
print(f"N = {int(aux_reg.nobs)},  R² = {aux_reg.rsquared:.4f},  R²_adj = {aux_reg.rsquared_adj:.4f}")

### Q3(a) Constructing the Mean Family Marriage Rate and Verifying the Auxiliary Regression

We define the mean family marriage rate for each family $ f $ as the average of the twins’ marriage indicators:


$\overline{\text{married}}_f = \frac{1}{2} \sum_{i=1}^{2} \text{married}_{fi}$,

which takes values $ 0 $, $ \tfrac{1}{2} $, or $ 1 $, depending on whether neither, one, or both twins are married.

We then estimated the auxiliary regression:
$
\text{married}_{fi} = \alpha + \rho\,\overline{\text{married}}_f + \varepsilon_{fi}.
$

| Variable | Coefficient | Robust SE (HC1) | t | p \> \|t\| |  
|-----------|-------------:|---------------:|---------------:|---------------:|  
| Intercept | −0.0000 | 0.0074 | −0.0000 | 1.0000 |  
| Mean Married ($\overline{\text{married}}_f$) | 1.0000 | 0.0009 | 1147.51 | 0.0000 |  
|  |  |  |  |  |  
| **N = 1074**  **R² = 0.737**  **R²_adj = 0.737** |  |  |  |  |


**Interpretation.**  
The coefficient on the family mean marriage rate is essentially 1, confirming the theoretical result proven in Lecture 8.  
When a binary variable is regressed on its own group mean (for groups of equal size), the estimated slope must equal 1 because the group mean perfectly summarizes within-group variation in the dependent variable (I think--I am still very shakey on this idea).  

Intuitively, a twin’s marriage status is mechanically linked to the family’s mean marriage rate—if one twin’s marital status changes, the family mean changes in the same direction by exactly half, producing a one-to-one relationship in the regression.

### (b) 
Add mean fraction of siblings married to your gender-specific wage models from part 2. How does the addition of this variable affect the estimated coefficient on marriage. Give an interpretation of the patterns and how they differ between men and women. 

In [ ]:
# --- Q3(b): Add mean_married to the gender-specific models (HC1 robust) ---

import statsmodels.formula.api as smf
import pandas as pd
import numpy as np

# If part-2 objects aren't in memory, quickly recreate thems
formula_q2 = "lw ~ educ + exp + exp2 + married"
try:
    ols_men, ols_women
except NameError:
    men    = twins.loc[twins["female"] == 0].copy()
    women  = twins.loc[twins["female"] == 1].copy()
    ols_men    = smf.ols(formula_q2, data=men).fit(cov_type="HC1")
    ols_women  = smf.ols(formula_q2, data=women).fit(cov_type="HC1")

# New models WITH the family mean marriage rate
formula_q3b = "lw ~ educ + exp + exp2 + married + mean_married"

men    = twins.loc[twins["female"] == 0].copy()
women  = twins.loc[twins["female"] == 1].copy()

ols_men_b   = smf.ols(formula_q3b, data=men).fit(cov_type="HC1")
ols_women_b = smf.ols(formula_q3b, data=women).fit(cov_type="HC1")

def tidy(res):
    return (pd.DataFrame({
        "coef": res.params,
        "se(HC1)": res.bse,
        "t": res.tvalues,
        "p>|t|": res.pvalues
    }).round(4))

print(f"Men — N={int(ols_men_b.nobs)}, R²={ols_men_b.rsquared:.4f}, R²_adj={ols_men_b.rsquared_adj:.4f}")
display(tidy(ols_men_b).loc[["Intercept","educ","exp","exp2","married","mean_married"]])

print(f"\nWomen — N={int(ols_women_b.nobs)}, R²={ols_women_b.rsquared:.4f}, R²_adj={ols_women_b.rsquared_adj:.4f}")
display(tidy(ols_women_b).loc[["Intercept","educ","exp","exp2","married","mean_married"]])

In [ ]:
# --- Compare 'married' coefficient before (Q2) vs after adding mean_married (Q3b) ---

def coef(res, name):
    return float(res.params.get(name, np.nan))

cmp = pd.DataFrame({
    "group": ["Men","Women"],
    "married_Q2": [coef(ols_men, "married"), coef(ols_women, "married")],
    "married_Q3b": [coef(ols_men_b, "married"), coef(ols_women_b, "married")],
    "mean_married_coef": [coef(ols_men_b, "mean_married"), coef(ols_women_b, "mean_married")]
})
cmp["Δ married (Q3b − Q2)"] = (cmp["married_Q3b"] - cmp["married_Q2"])
display(cmp.round(4))

Here’s a clean Markdown cell you can paste right under your Q3(b) code output. It uses inline math and a tiny comparison table.

---

### Q3(b) Adding the family mean marriage rate to gender-specific wage models

We augment the Q2 models to include the family mean marriage rate $ \overline{\text{married}}_f $ and re-estimate
$
\text{lw}_i = \beta_0+\beta_1 \text{educ}_i+\beta_2 \text{exp}_i+\beta_3 \text{exp}_i^2+\beta_4 \text{married}_i+\beta_5 \overline{\text{married}}_f+u_i,
$
separately for men and women (HC1 SEs).

**Key coefficients (marriage terms only):**

| Group | $ \hat\beta_4 $ on $ \text{married}_i $ (Q2) | $ \hat\beta_4 $ on $ \text{married}_i $ (Q3b) | $ \hat\beta_5 $ on $ \overline{\text{married}}_f $ (Q3b) | Δ married (Q3b − Q2) |
|---|---:|---:|---:|---:|
| Men | 0.1993 | 0.0300 | 0.2500 | −0.1693 |
| Women | −0.0399 | −0.0221 | −0.0268 | 0.0179 |

**What changed?**  
- **Men.** The individual marriage premium shrinks from $ \approx 0.199 $ to $ \approx 0.030 $ and loses significance, while the family mean term is positive and significant (≈ 0.25).  
- **Women.** The individual marriage coefficient was already small and insignificant (≈ −0.040) and remains small and insignificant after adding the family mean; the family mean term is also small and insignificant.

**Interpretation.**  
- With two-twin families, $ \overline{\text{married}}_f=(\text{married}_i+\text{married}_j)/2 $. Including $ \overline{\text{married}}_f $ absorbs the **family-level (shared) component** of marriage—selection or factors common to both twins (values, timing, labor supply norms, etc.).  
- For **men**, most of the apparent marriage premium in Q2 is captured by this shared component (strong $ \hat\beta_5>0 $), leaving little **individual-specific** premium once family mean is controlled. This pattern is consistent with positive selection into marriage among men rather than a large individual causal effect.  
- For **women**, there is no evident premium to begin with, and controlling for family mean changes nothing—both the individual and family components are economically small and statistically insignificant.

**Model fit.** $R^2$ barely moves for women (≈ 0.303 → 0.303) and rises slightly for men (≈ 0.255 → 0.262), consistent with the family mean having explanatory power mainly in the male sample.

### (c)
Instead of controlling for the mean family marriage rate, estimate the gender-specific wage models from part 2 with a control for the other twin's marriage status (variable “omarried”). Compare the regression coefficients on (married, omarried) with the coefficient on (married, mean married) in (b). 

In [ ]:
# --- Q3(c): Add 'omarried' (other twin married) instead of family mean; compare to Q3(b) ---

import statsmodels.formula.api as smf
import pandas as pd
import numpy as np

# If Q2/Q3b results aren't in memory, (re)create them quickly so comparisons work
formula_q2  = "lw ~ educ + exp + exp2 + married"
men    = twins.loc[twins["female"] == 0].copy()
women  = twins.loc[twins["female"] == 1].copy()

try:
    _ = ols_men, ols_women   # Q2
except NameError:
    ols_men   = smf.ols(formula_q2, data=men).fit(cov_type="HC1")
    ols_women = smf.ols(formula_q2, data=women).fit(cov_type="HC1")

try:
    _ = ols_men_b, ols_women_b  # Q3(b): had 'mean_married'
except NameError:
    # Make sure mean_married exists
    if "mean_married" not in twins.columns:
        twins["mean_married"] = twins.groupby("famid")["married"].transform("mean")
        men   = twins.loc[twins["female"] == 0].copy()
        women = twins.loc[twins["female"] == 1].copy()
    formula_q3b = "lw ~ educ + exp + exp2 + married + mean_married"
    ols_men_b   = smf.ols(formula_q3b, data=men).fit(cov_type="HC1")
    ols_women_b = smf.ols(formula_q3b, data=women).fit(cov_type="HC1")

# ---- New Q3(c) models: add 'omarried' (other twin's marriage) ----
formula_q3c = "lw ~ educ + exp + exp2 + married + omarried"
ols_men_c   = smf.ols(formula_q3c, data=men).fit(cov_type="HC1")
ols_women_c = smf.ols(formula_q3c, data=women).fit(cov_type="HC1")

def tidy(res):
    return (pd.DataFrame({
        "coef":   res.params,
        "se(HC1)":res.bse,
        "t":      res.tvalues,
        "p>|t|":  res.pvalues
    }).round(4))

print(f"Men  — N={int(ols_men_c.nobs)}, R²={ols_men_c.rsquared:.4f}, R²_adj={ols_men_c.rsquared_adj:.4f}")
display(tidy(ols_men_c).loc[["Intercept","educ","exp","exp2","married","omarried"]])

print(f"\nWomen— N={int(ols_women_c.nobs)}, R²={ols_women_c.rsquared:.4f}, R²_adj={ols_women_c.rsquared_adj:.4f}")
display(tidy(ols_women_c).loc[["Intercept","educ","exp","exp2","married","omarried"]])

# ---- Side-by-side comparison with Q3(b) (mean_married) and Q2 (baseline) ----
def coef(res, name):  # safe getter
    return float(res.params.get(name, np.nan))

cmp_c = pd.DataFrame({
    "group": ["Men","Women"],
    # Q2 (baseline, by sex)
    "married_Q2":        [coef(ols_men, "married"),        coef(ols_women, "married")],
    # Q3(b) (with mean_married)
    "married_Q3b":       [coef(ols_men_b, "married"),      coef(ols_women_b, "married")],
    "mean_married_Q3b":  [coef(ols_men_b, "mean_married"), coef(ols_women_b, "mean_married")],
    # Q3(c) (with omarried)
    "married_Q3c":       [coef(ols_men_c, "married"),      coef(ols_women_c, "married")],
    "omarried_Q3c":      [coef(ols_men_c, "omarried"),     coef(ols_women_c, "omarried")],
})

# Helpful deltas
cmp_c["Δ married (Q3c − Q2)"] = (cmp_c["married_Q3c"] - cmp_c["married_Q2"]).round(4)
cmp_c["Δ married (Q3c − Q3b)"] = (cmp_c["married_Q3c"] - cmp_c["married_Q3b"]).round(4)

display(cmp_c.round(4))

### Q3(c) Comparing the “Other Twin Married” and “Mean Married” Models

Replacing the family mean marriage rate $ \overline{\text{married}}_f $ with the other twin’s marriage status $ \text{omarried}_i $, we estimate:  
$
\text{lw}_i = \beta_0 + \beta_1 \text{educ}_i + \beta_2 \text{exp}_i + \beta_3 \text{exp}_i^2 + \beta_4 \text{married}_i + \beta_5 \text{omarried}_i + u_i
$

For men, the coefficient on own marriage increases to $ 0.155 $ while the co-twin’s marriage coefficient is $ 0.125 $. Together, their combined effect ($ 0.155 + 0.125 \approx 0.28 $) is roughly equal to the family mean coefficient ($ 0.25 $) from part (b). This makes sense since $ \overline{\text{married}}_f = \tfrac{1}{2}(\text{married}_i + \text{omarried}_i) $, meaning the “mean married” variable in (b) effectively averages the two twins’ statuses. For women, both coefficients remain small and statistically insignificant, similar to before.  

Overall, controlling for the co-twin’s marriage shows that the male marriage premium mainly reflects family or shared factors rather than purely individual effects, while for women, marriage continues to have no meaningful association with wages.

## 4. Three ways to get the within estimator: 
In lecture 8 we show three ways to get the within estimator - de-meaned relative to the mean, fixed effects, and control function. Let's run 3 regressions for **male** twins. 

### (a) 
De-mean variables relative to the mean in each family: replace $x_{it}$ by $x_{it}-\bar{x}_{i}$ where $x_{it}$ includes education, experience-squared, and married status, and replace $y_{it}$ by $y_{it}-\bar{y}_{i}$. Fit a regression of $(y_{it}-\bar{y}_{i})$ on $x_{it}-\bar{x}_{i}$. 
- Note we dropped experience from the regression. See how experience is defined. The demeaned experience = - demeaned education (correlation = -1). 

In [ ]:
keys = ['famid','t'] # twins data unique at (family, twin) level
select= ['lw','educ','exp','exp2','married']
# focus on families with male twins:
fam_means = twins.loc[twins['female']==0].groupby(['famid'])[select].mean().reset_index(drop=False)
fam_means.columns=['famid'] + [f'mean_{x}' for x in select]

# merge fam_means with twins (note you may have defined mean marriage rate already) by famid
male_twins = pd.merge(twins.loc[twins['female']==0, keys+select], fam_means, on=['famid'],how='left')

# compute the demeaned x and y: 
for z in select:
    male_twins[f'd_{z}'] = male_twins[z] - male_twins[f'mean_{z}'] 

# note: d_exp = - d_educ. exp is defined as age - educ - 6. Within each family, twins are of the same age, d_exp = exp - (age  - mean_educ -6) = - educ + mean_educ = - d_educ
print(male_twins[['d_exp','d_educ']].corr())
assert (abs(male_twins['d_exp'] + male_twins['d_educ'])<1e-12).all()

In [ ]:
# OLS for within: formula = "d_lw ~ d_educ + d_exp + d_exp2 + d_married" 


### (b) Fixed Effects
Fit a regression of y_{it} on x_{it} and family fixed effects. 
- *Note if you use PanelOLS from linearmodels, make sure to drop collinear variables before adding “entityeffects”. For example, educ and exp are collinear with each other conditional on family fixed effects (twins have the same age). We have dropped exp from the regression.*

In [ ]:
# may consider: 
#  pip install linearmodels 
#  from linearmodels import PanelOLS

### (c) Control Function
Fit a regression of $y_{it}$ on $x_{it}$ and means $\bar{x}_{i}$ in each family. Verify if the coefficients on $x_{it}$ are the same as in (a) and (b). 